In [1]:
import warnings
from causalnex.structure import StructureModel
warnings.filterwarnings("ignore") 
from causalnex.structure.notears import from_pandas
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.evaluation import roc_auc
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx

In [2]:
def calculate_bic(bn, data):
    """
    Calculates BIC score for a CausalNex BayesianNetwork.
    Formula: BIC = k * ln(n) - 2 * ln(L)
    Returns: bic, log_likelihood
    """
    log_likelihood = 0
    k = 0
    n = len(data)

    for node in bn.nodes:
        cpd = bn.cpds[node]
        
        probs = bn.predict_probability(data, node)
        
        true_values = data[node]
        
        y_dummies = pd.get_dummies(true_values)
        
        y_dummies = y_dummies.reindex(columns=probs.columns, fill_value=0)
        
        p_actual = (probs * y_dummies).sum(axis=1)
        
        log_likelihood += np.sum(np.log(p_actual + 1e-9))

        num_parent_configs = cpd.shape[0]
        num_node_states = cpd.shape[1]
        k += num_parent_configs * (num_node_states - 1)

    bic = k * np.log(n) - 2 * log_likelihood
    
    return bic, log_likelihood

In [3]:
from sklearn.mixture import GaussianMixture # <--- NEW IMPORT
from sklearn.preprocessing import StandardScaler
import itertools

def add_latent_variable_gmm(bn_baseline, sm_baseline, children_cols, discretised_data, raw_data=None):
    
    if raw_data is None:
        raise Valuserror("You must provide the continuous 'raw_data' to accurately estimate the Latent Variable.")
        
    subset_data = raw_data[children_cols].loc[discretised_data.index].copy()
    
    
    print(f"Estimating LV based on continuous GMM distributions in: {children_cols}")

    scaler = StandardScaler()
    subset_data_scaled = scaler.fit_transform(subset_data)
    
    gmm = GaussianMixture(n_components=2, random_state=42, n_init=10)
    proxy_lv = gmm.fit_predict(subset_data_scaled)
    
    discretised_with_lv = discretised_data.copy()
    discretised_with_lv['LV'] = proxy_lv
    discretised_with_lv['LV'] = discretised_with_lv['LV'].map({0: "State0", 1: "State1"})
    
    sm_with_lv = sm_baseline.copy()
    sm_with_lv.add_node('LV')
    
    for child in children_cols:
        sm_with_lv.add_edge('LV', child)
        
    for u, v in itertools.permutations(children_cols, 2):
        if sm_with_lv.has_edge(u, v):
            print(f"  - Removing direct edge: {u} -> {v}")
            sm_with_lv.remove_edge(u, v)

    bn_with_lv = BayesianNetwork(sm_with_lv)
    bn_with_lv.fit_node_states(discretised_with_lv)
    bn_with_lv.fit_cpds(discretised_with_lv, method="BayesianEstimator", bayes_prior="K2")
    
    bic_lv, ll_lv = calculate_bic(bn_with_lv, discretised_with_lv)
    print(f"LV Model BIC: {bic_lv:.2f} (LogLikelihood: {ll_lv:.2f})")
    
    return bn_with_lv, discretised_with_lv, bic_lv

In [4]:
df_se = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/se_bn_no_context.csv")

df_se = df_se.drop(["Unnamed: 0"], axis = 1)

df_se

,GDPC,VABI,EMP,QOA,QOR,YUA,REST,EBSG,GGFC,LBGD,FDI,PC,ARP,CO2C,PAM,EUPC
0,362059.214096,25.02,79.7,6.030000,5.65,19.26,43.224,5.900063,24.205811,3.214195,9.42,7.6,10.5,0.000006,276.174863,3.554298
1,367898.323980,24.15,80.0,6.020000,5.63,20.21,43.922,5.218714,24.785451,2.032230,8.05,8.0,13.5,0.000006,276.464208,5.290971
2,357492.674906,22.33,77.9,6.040000,5.69,25.02,47.024,4.958800,26.132116,-0.614954,2.06,9.1,14.4,0.000005,235.053763,19.503994
3,378703.698372,23.86,77.7,6.200000,5.71,24.76,46.099,4.661454,25.187785,-0.240938,0.13,8.0,14.8,0.000006,234.115139,18.684230
4,392078.684225,23.73,79.0,6.250000,5.66,22.78,47.632,4.436392,25.092481,0.015331,1.22,7.1,15.4,0.000005,212.063492,33.708495
5,391255.979647,23.11,79.1,5.970000,5.58,23.61,49.403,4.367928,25.878585,-0.914026,0.78,7.7,15.2,0.000005,240.336134,11.559979
6,396337.894577,21.98,79.4,5.730000,5.52,23.52,50.153,3.831176,26.318281,-1.371356,0.25,9.3,16.0,0.000005,242.916667,26.080148
7,408791.051257,21.61,79.6,5.670000,5.50,22.82,51.151,3.093335,26.275487,-1.129253,-1.05,10.6,15.6,0.000004,204.536082,35.538469
8,431846.583992,22.28,80.1,5.600000,5.36,20.36,52.220,3.259861,25.892770,0.377740,1.95,10.6,16.3,0.000004,207.959184,8.000893
9,443685.809403,21.96,80.8,5.700000,5.30,18.90,52.597,2.557094,26.439038,1.576112,3.01,14.5,16.2,0.000004,204.838710,5.269573


In [5]:
se = pd.DataFrame()
se = df_se.copy()
se['QOR_2'] = se['QOR'].shift(2)
se['QOA_2'] = se['QOA'].shift(2)
se['GGFC_3'] = se['GGFC'].shift(3)

In [6]:
se = se.dropna()

In [7]:
discretised_se = pd.DataFrame(index=se.index)

for col in se.columns:
    no_unique = se[col].nunique()
    
    if no_unique <= 1:
        discretised_se[col] = 0
    else:
        try:
            discretised_se[col] = pd.qcut(
                se[col], 
                q=min(2, no_unique),  
                labels=False,
                duplicates='drop'
            ).astype(int)
        except Valuserror:
            discretised_se[col] = se[col].rank(method='sense').astype(int) - 1
    num_states = discretised_se[col].nunique()
    
    if num_states == 2:
        discretised_se[col] = discretised_se[col].map({0: "Low", 1: "High"})
    elif num_states == 3:
        discretised_se[col] = discretised_se[col].map({0: "Low", 1: "Medium", 2: "High"})
    else:
        discretised_se[col] = discretised_se[col].astype(str)

print("Discretised data:")
print(discretised_se)

Discretised data:
    GDPC  VABI   EMP   QOA   QOR   YUA  REST  EBSG  GGFC  LBGD   FDI    PC  \
3    Low  High   Low  High  High  High   Low  High   Low   Low   Low   Low   
4    Low  High   Low  High  High   Low   Low  High   Low   Low   Low   Low   
5    Low  High   Low  High  High  High   Low  High   Low   Low   Low   Low   
6    Low   Low   Low   Low  High  High   Low   Low  High   Low   Low   Low   
7    Low   Low   Low   Low   Low  High   Low   Low  High   Low   Low  High   
8    Low   Low   Low   Low   Low   Low   Low   Low   Low  High   Low  High   
9    Low   Low  High   Low   Low   Low   Low   Low  High  High  High  High   
10  High   Low  High   Low   Low   Low  High   Low  High  High  High  High   
11  High   Low  High   Low  High   Low  High   Low  High  High   Low  High   
12  High  High  High   Low   Low   Low  High  High   Low  High  High  High   
13  High   Low   Low  High   Low  High  High  High  High   Low  High   Low   
14  High  High  High  High   Low  High  High  

In [8]:
sm = StructureModel()

In [9]:
sm.add_edges_from([  
    ('EUPC', 'QOA'),
     ('EUPC', 'FDI'),
     ('QOR_2', 'FDI'),
     ('QOR_2', 'GDPC'),
    ('GGFC_3', 'EUPC'),
    ('GDPC', 'ARP'),
    ('QOR', 'ARP'),
    ('ARP', 'GGFC'),
])



In [10]:
sm.edges

OutEdgeView([('EUPC', 'QOA'), ('EUPC', 'FDI'), ('QOR_2', 'FDI'), ('QOR_2', 'GDPC'), ('GDPC', 'ARP'), ('GGFC_3', 'EUPC'), ('ARP', 'GGFC'), ('QOR', 'ARP')])

In [11]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/fully_connected_dk_1.html")

Graphs/fully_connected_dk_1.html


In [12]:
bn = BayesianNetwork(sm)

bn.fit_node_states(discretised_se)

bn.fit_cpds(discretised_se, method="BayesianEstimator", bayes_prior="K2")

bic_baseline, ll_baseline = calculate_bic(bn, discretised_se)
print(f"Baseline BIC: {bic_baseline:.2f} (LogLikelihood: {ll_baseline:.2f})")

Baseline BIC: 4900.54 (LogLikelihood: -2424.62)


In [13]:
bn_with_lv, data_with_lv, bic_lv = add_latent_variable_gmm(
    bn_baseline=bn,
    sm_baseline=sm,
    children_cols=['EUPC', 'QOA'],  # Replace with your specific nodes
    discretised_data=discretised_se,
    raw_data=se
)

Estimating LV based on continuous GMM distributions in: ['EUPC', 'QOA']
  - Removing direct edge: EUPC -> QOA
LV Model BIC: 5449.61 (LogLikelihood: -2694.02)


In [14]:
bn_with_lv_1, data_with_lv_1, bic_lv_1 = add_latent_variable_gmm(
    bn_baseline=bn,
    sm_baseline=sm,
    children_cols=['EUPC', 'FDI'],  # Replace with your specific nodes
    discretised_data=discretised_se,
    raw_data=se
)

Estimating LV based on continuous GMM distributions in: ['EUPC', 'FDI']
  - Removing direct edge: EUPC -> FDI
LV Model BIC: 5449.61 (LogLikelihood: -2694.02)


In [15]:
bn_with_lv_1, data_with_lv_1, bic_lv_1 = add_latent_variable_gmm(
    bn_baseline=bn,
    sm_baseline=sm,
    children_cols=['GGFC_3', 'EUPC'], 
    discretised_data=discretised_se,
    raw_data=se
)

Estimating LV based on continuous GMM distributions in: ['GGFC_3', 'EUPC']
  - Removing direct edge: GGFC_3 -> EUPC
LV Model BIC: 5444.48 (LogLikelihood: -2694.02)


In [4]:
import warnings
from causalnex.structure import StructureModel
warnings.filterwarnings("ignore") 
from causalnex.structure.notears import from_pandas
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.evaluation import roc_auc
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx


In [5]:
df_se = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/se_bn_no_context.csv")

df_se = df_se.drop(["Unnamed: 0"], axis = 1)

df_se

,GDPC,VABI,EMP,QOA,QOR,YUA,REST,EBSG,GGFC,LBGD,FDI,PC,ARP,CO2C,PAM,EUPC
0,362059.214096,25.02,79.7,6.030000,5.65,19.26,43.224,5.900063,24.205811,3.214195,9.42,7.6,10.5,0.000006,276.174863,3.554298
1,367898.323980,24.15,80.0,6.020000,5.63,20.21,43.922,5.218714,24.785451,2.032230,8.05,8.0,13.5,0.000006,276.464208,5.290971
2,357492.674906,22.33,77.9,6.040000,5.69,25.02,47.024,4.958800,26.132116,-0.614954,2.06,9.1,14.4,0.000005,235.053763,19.503994
3,378703.698372,23.86,77.7,6.200000,5.71,24.76,46.099,4.661454,25.187785,-0.240938,0.13,8.0,14.8,0.000006,234.115139,18.684230
4,392078.684225,23.73,79.0,6.250000,5.66,22.78,47.632,4.436392,25.092481,0.015331,1.22,7.1,15.4,0.000005,212.063492,33.708495
5,391255.979647,23.11,79.1,5.970000,5.58,23.61,49.403,4.367928,25.878585,-0.914026,0.78,7.7,15.2,0.000005,240.336134,11.559979
6,396337.894577,21.98,79.4,5.730000,5.52,23.52,50.153,3.831176,26.318281,-1.371356,0.25,9.3,16.0,0.000005,242.916667,26.080148
7,408791.051257,21.61,79.6,5.670000,5.50,22.82,51.151,3.093335,26.275487,-1.129253,-1.05,10.6,15.6,0.000004,204.536082,35.538469
8,431846.583992,22.28,80.1,5.600000,5.36,20.36,52.220,3.259861,25.892770,0.377740,1.95,10.6,16.3,0.000004,207.959184,8.000893
9,443685.809403,21.96,80.8,5.700000,5.30,18.90,52.597,2.557094,26.439038,1.576112,3.01,14.5,16.2,0.000004,204.838710,5.269573


In [6]:
se = pd.DataFrame()
se = df_se.copy()
se['EUPC_2'] = se['EUPC'].shift(2)

In [7]:
se = se.dropna()

In [8]:
discretised_se = pd.DataFrame(index=se.index)

for col in se.columns:
    no_unique = se[col].nunique()
    
    if no_unique <= 1:
        discretised_se[col] = 0
    else:
        try:
            discretised_se[col] = pd.qcut(
                se[col], 
                q=min(2, no_unique),  
                labels=False,
                duplicates='drop'
            ).astype(int)
        except Valuserror:
            discretised_se[col] = se[col].rank(method='sense').astype(int) - 1
    num_states = discretised_se[col].nunique()
    
    if num_states == 2:
        discretised_se[col] = discretised_se[col].map({0: "Low", 1: "High"})
    elif num_states == 3:
        discretised_se[col] = discretised_se[col].map({0: "Low", 1: "Medium", 2: "High"})
    else:
        discretised_se[col] = discretised_se[col].astype(str)

print("Discretised data:")
print(discretised_se)

Discretised data:
    GDPC  VABI   EMP   QOA   QOR   YUA  REST  EBSG  GGFC  LBGD   FDI    PC  \
2    Low  High   Low  High  High  High   Low  High  High   Low  High   Low   
3    Low  High   Low  High  High  High   Low  High   Low   Low   Low   Low   
4    Low  High   Low  High  High   Low   Low  High   Low  High   Low   Low   
5    Low  High   Low  High  High  High   Low  High   Low   Low   Low   Low   
6    Low   Low   Low   Low  High  High   Low   Low  High   Low   Low  High   
7    Low   Low   Low   Low   Low  High   Low   Low  High   Low   Low  High   
8    Low   Low   Low   Low   Low   Low   Low   Low   Low  High   Low  High   
9   High   Low  High   Low   Low   Low  High   Low  High  High  High  High   
10  High   Low  High   Low   Low   Low  High   Low  High  High  High  High   
11  High   Low  High   Low  High   Low  High   Low  High  High   Low  High   
12  High  High  High   Low   Low   Low  High  High   Low  High  High  High   
13  High   Low  High  High   Low  High  High  

In [9]:
sm = StructureModel()

In [10]:
sm.add_edges_from([
    ('EUPC_2', 'QOR'),
    ('EUPC_2', 'EBSG'),
    ('QOR', 'ARP'),
    ('GDPC', 'ARP')
])

In [11]:
sm.edges

OutEdgeView([('EUPC_2', 'QOR'), ('EUPC_2', 'EBSG'), ('QOR', 'ARP'), ('GDPC', 'ARP')])

In [12]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/fully_connected_dk_2.html")

Graphs/fully_connected_dk_2.html


In [13]:
bn = BayesianNetwork(sm)

bn.fit_node_states(discretised_se)

bn.fit_cpds(discretised_se, method="BayesianEstimator", bayes_prior="K2")

bic_baseline, ll_baseline = calculate_bic(bn, discretised_se)
print(f"Baseline BIC: {bic_baseline:.2f} (LogLikelihood: {ll_baseline:.2f})")

Baseline BIC: 2927.65 (LogLikelihood: -1450.63)


In [14]:
bn_with_lv_1, data_with_lv_1, bic_lv_1 = add_latent_variable_gmm(
    bn_baseline=bn,
    sm_baseline=sm,
    children_cols=['EUPC_2', 'QOR'],  # Replace with your specific nodes
    discretised_data=discretised_se,
    raw_data=se
)

Estimating LV based on continuous GMM distributions in: ['EUPC_2', 'QOR']
  - Removing direct edge: EUPC_2 -> QOR
LV Model BIC: 3513.18 (LogLikelihood: -1740.75)


In [16]:
bn_with_lv_2, data_with_lv_2, bic_lv_2 = add_latent_variable_gmm(
    bn_baseline=bn,
    sm_baseline=sm,
    children_cols=['EUPC_2', 'EBSG'], 
    discretised_data=discretised_se,
    raw_data=se
)

Estimating LV based on continuous GMM distributions in: ['EUPC_2', 'EBSG']
  - Removing direct edge: EUPC_2 -> EBSG
LV Model BIC: 3513.18 (LogLikelihood: -1740.75)


In [17]:
import warnings
from causalnex.structure import StructureModel
warnings.filterwarnings("ignore") 
from causalnex.structure.notears import from_pandas
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.evaluation import roc_auc
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx


In [18]:
df_se = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/se_bn_no_context.csv")

df_se = df_se.drop(["Unnamed: 0"], axis = 1)

df_se

,GDPC,VABI,EMP,QOA,QOR,YUA,REST,EBSG,GGFC,LBGD,FDI,PC,ARP,CO2C,PAM,EUPC
0,362059.214096,25.02,79.7,6.030000,5.65,19.26,43.224,5.900063,24.205811,3.214195,9.42,7.6,10.5,0.000006,276.174863,3.554298
1,367898.323980,24.15,80.0,6.020000,5.63,20.21,43.922,5.218714,24.785451,2.032230,8.05,8.0,13.5,0.000006,276.464208,5.290971
2,357492.674906,22.33,77.9,6.040000,5.69,25.02,47.024,4.958800,26.132116,-0.614954,2.06,9.1,14.4,0.000005,235.053763,19.503994
3,378703.698372,23.86,77.7,6.200000,5.71,24.76,46.099,4.661454,25.187785,-0.240938,0.13,8.0,14.8,0.000006,234.115139,18.684230
4,392078.684225,23.73,79.0,6.250000,5.66,22.78,47.632,4.436392,25.092481,0.015331,1.22,7.1,15.4,0.000005,212.063492,33.708495
5,391255.979647,23.11,79.1,5.970000,5.58,23.61,49.403,4.367928,25.878585,-0.914026,0.78,7.7,15.2,0.000005,240.336134,11.559979
6,396337.894577,21.98,79.4,5.730000,5.52,23.52,50.153,3.831176,26.318281,-1.371356,0.25,9.3,16.0,0.000005,242.916667,26.080148
7,408791.051257,21.61,79.6,5.670000,5.50,22.82,51.151,3.093335,26.275487,-1.129253,-1.05,10.6,15.6,0.000004,204.536082,35.538469
8,431846.583992,22.28,80.1,5.600000,5.36,20.36,52.220,3.259861,25.892770,0.377740,1.95,10.6,16.3,0.000004,207.959184,8.000893
9,443685.809403,21.96,80.8,5.700000,5.30,18.90,52.597,2.557094,26.439038,1.576112,3.01,14.5,16.2,0.000004,204.838710,5.269573


In [19]:
se = pd.DataFrame()
se = df_se.copy()
se['EUPC_1'] = se['EUPC'].shift(1)

In [20]:
se = se.dropna()

In [21]:
discretised_se = pd.DataFrame(index=se.index)

for col in se.columns:
    no_unique = se[col].nunique()
    
    if no_unique <= 1:
        discretised_se[col] = 0
    else:
        try:
            discretised_se[col] = pd.qcut(
                se[col], 
                q=min(2, no_unique),  
                labels=False,
                duplicates='drop'
            ).astype(int)
        except Valuserror:
            discretised_se[col] = se[col].rank(method='sense').astype(int) - 1
    num_states = discretised_se[col].nunique()
    
    if num_states == 2:
        discretised_se[col] = discretised_se[col].map({0: "Low", 1: "High"})
    elif num_states == 3:
        discretised_se[col] = discretised_se[col].map({0: "Low", 1: "Medium", 2: "High"})
    else:
        discretised_se[col] = discretised_se[col].astype(str)

print("Discretised data:")
print(discretised_se)

Discretised data:
    GDPC  VABI   EMP   QOA   QOR   YUA  REST  EBSG  GGFC  LBGD   FDI    PC  \
1    Low  High   Low  High  High   Low   Low  High   Low  High  High   Low   
2    Low  High   Low  High  High  High   Low  High  High   Low   Low   Low   
3    Low  High   Low  High  High  High   Low  High   Low   Low   Low   Low   
4    Low  High   Low  High  High   Low   Low  High   Low   Low   Low   Low   
5    Low  High   Low  High  High  High   Low  High   Low   Low   Low   Low   
6    Low   Low   Low   Low  High  High   Low   Low  High   Low   Low  High   
7    Low   Low   Low   Low   Low  High   Low   Low  High   Low   Low  High   
8    Low   Low   Low   Low   Low   Low   Low   Low   Low  High   Low  High   
9   High   Low  High   Low   Low   Low  High   Low  High  High  High  High   
10  High   Low  High   Low   Low   Low  High   Low  High  High  High  High   
11  High   Low  High   Low  High   Low  High   Low  High  High   Low  High   
12  High   Low  High   Low   Low   Low  High  

In [22]:
sm = StructureModel()

In [23]:
sm.add_edges_from([
    ('EUPC_1', 'PC')
])

In [24]:
sm.edges

OutEdgeView([('EUPC_1', 'PC')])

In [25]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/fully_connected_dk_2.html")

Graphs/fully_connected_dk_2.html


In [26]:
bn = BayesianNetwork(sm)

bn.fit_node_states(discretised_se)

bn.fit_cpds(discretised_se, method="BayesianEstimator", bayes_prior="K2")

bic_baseline, ll_baseline = calculate_bic(bn, discretised_se)
print(f"Baseline BIC: {bic_baseline:.2f} (LogLikelihood: {ll_baseline:.2f})")

Baseline BIC: 1248.81 (LogLikelihood: -621.70)


In [27]:
bn_with_lv_1, data_with_lv_1, bic_lv_1 = add_latent_variable_gmm(
    bn_baseline=bn,
    sm_baseline=sm,
    children_cols=['EUPC_1', 'PC'],  # Replace with your specific nodes
    discretised_data=discretised_se,
    raw_data=se
)

Estimating LV based on continuous GMM distributions in: ['EUPC_1', 'PC']
  - Removing direct edge: EUPC_1 -> PC
LV Model BIC: 1875.93 (LogLikelihood: -932.55)


In [32]:
import warnings
from causalnex.structure import StructureModel
warnings.filterwarnings("ignore") 
from causalnex.structure.notears import from_pandas
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.evaluation import roc_auc
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx


In [33]:
df_se = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/se_bn_no_context.csv")

df_se = df_se.drop(["Unnamed: 0"], axis = 1)

df_se

,GDPC,VABI,EMP,QOA,QOR,YUA,REST,EBSG,GGFC,LBGD,FDI,PC,ARP,CO2C,PAM,EUPC
0,362059.214096,25.02,79.7,6.030000,5.65,19.26,43.224,5.900063,24.205811,3.214195,9.42,7.6,10.5,0.000006,276.174863,3.554298
1,367898.323980,24.15,80.0,6.020000,5.63,20.21,43.922,5.218714,24.785451,2.032230,8.05,8.0,13.5,0.000006,276.464208,5.290971
2,357492.674906,22.33,77.9,6.040000,5.69,25.02,47.024,4.958800,26.132116,-0.614954,2.06,9.1,14.4,0.000005,235.053763,19.503994
3,378703.698372,23.86,77.7,6.200000,5.71,24.76,46.099,4.661454,25.187785,-0.240938,0.13,8.0,14.8,0.000006,234.115139,18.684230
4,392078.684225,23.73,79.0,6.250000,5.66,22.78,47.632,4.436392,25.092481,0.015331,1.22,7.1,15.4,0.000005,212.063492,33.708495
5,391255.979647,23.11,79.1,5.970000,5.58,23.61,49.403,4.367928,25.878585,-0.914026,0.78,7.7,15.2,0.000005,240.336134,11.559979
6,396337.894577,21.98,79.4,5.730000,5.52,23.52,50.153,3.831176,26.318281,-1.371356,0.25,9.3,16.0,0.000005,242.916667,26.080148
7,408791.051257,21.61,79.6,5.670000,5.50,22.82,51.151,3.093335,26.275487,-1.129253,-1.05,10.6,15.6,0.000004,204.536082,35.538469
8,431846.583992,22.28,80.1,5.600000,5.36,20.36,52.220,3.259861,25.892770,0.377740,1.95,10.6,16.3,0.000004,207.959184,8.000893
9,443685.809403,21.96,80.8,5.700000,5.30,18.90,52.597,2.557094,26.439038,1.576112,3.01,14.5,16.2,0.000004,204.838710,5.269573


In [34]:
se = pd.DataFrame()
se = df_se.copy()
se['QOA_2'] = se['QOA'].shift(2)

In [35]:
se = se.dropna()

In [36]:
discretised_se = pd.DataFrame(index=se.index)

for col in se.columns:
    no_unique = se[col].nunique()
    
    if no_unique <= 1:
        discretised_se[col] = 0
    else:
        try:
            discretised_se[col] = pd.qcut(
                se[col], 
                q=min(2, no_unique),  
                labels=False,
                duplicates='drop'
            ).astype(int)
        except Valuserror:
            discretised_se[col] = se[col].rank(method='sense').astype(int) - 1
    num_states = discretised_se[col].nunique()
    
    if num_states == 2:
        discretised_se[col] = discretised_se[col].map({0: "Low", 1: "High"})
    elif num_states == 3:
        discretised_se[col] = discretised_se[col].map({0: "Low", 1: "Medium", 2: "High"})
    else:
        discretised_se[col] = discretised_se[col].astype(str)

print("Discretised data:")
print(discretised_se)

Discretised data:
    GDPC  VABI   EMP   QOA   QOR   YUA  REST  EBSG  GGFC  LBGD   FDI    PC  \
2    Low  High   Low  High  High  High   Low  High  High   Low  High   Low   
3    Low  High   Low  High  High  High   Low  High   Low   Low   Low   Low   
4    Low  High   Low  High  High   Low   Low  High   Low  High   Low   Low   
5    Low  High   Low  High  High  High   Low  High   Low   Low   Low   Low   
6    Low   Low   Low   Low  High  High   Low   Low  High   Low   Low  High   
7    Low   Low   Low   Low   Low  High   Low   Low  High   Low   Low  High   
8    Low   Low   Low   Low   Low   Low   Low   Low   Low  High   Low  High   
9   High   Low  High   Low   Low   Low  High   Low  High  High  High  High   
10  High   Low  High   Low   Low   Low  High   Low  High  High  High  High   
11  High   Low  High   Low  High   Low  High   Low  High  High   Low  High   
12  High  High  High   Low   Low   Low  High  High   Low  High  High  High   
13  High   Low  High  High   Low  High  High  

In [37]:
sm = StructureModel()

In [38]:
sm.add_edges_from([
    ('QOA_2', 'EUPC')
])

In [39]:
sm.edges

OutEdgeView([('QOA_2', 'EUPC')])

In [40]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/fully_connected_dk_2.html")

Graphs/fully_connected_dk_2.html


In [41]:
bn = BayesianNetwork(sm)

bn.fit_node_states(discretised_se)

bn.fit_cpds(discretised_se, method="BayesianEstimator", bayes_prior="K2")

bic_baseline, ll_baseline = calculate_bic(bn, discretised_se)
print(f"Baseline BIC: {bic_baseline:.2f} (LogLikelihood: {ll_baseline:.2f})")

Baseline BIC: 1165.78 (LogLikelihood: -580.25)


In [42]:
bn_with_lv_1, data_with_lv_1, bic_lv_1 = add_latent_variable_gmm(
    bn_baseline=bn,
    sm_baseline=sm,
    children_cols=['QOA_2', 'EUPC'],  # Replace with your specific nodes
    discretised_data=discretised_se,
    raw_data=se
)

Estimating LV based on continuous GMM distributions in: ['QOA_2', 'EUPC']
  - Removing direct edge: QOA_2 -> EUPC
LV Model BIC: 1751.31 (LogLikelihood: -870.38)
